# Chapter 9: From Predictions to Potential Profit

This notebook is the canonical Chapter 9 notebook for the companion
code. It follows the chapter from odds mechanics and arbitrage through
baseline betting strategies, Kelly staking, and a simple machine-learning
betting workflow.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)
rng = np.random.default_rng(42)


## Soccer Betting Basics

We begin with the core mechanics behind betting markets: common market
types, payouts, implied probabilities, overround, and basic value-bet
logic.


In [ ]:
betting_markets = pd.DataFrame(
    {
        'Market': ['1X2', 'Over/Under 2.5', 'BTTS', 'Correct Score'],
        'Complexity': ['Low', 'Medium', 'Low', 'Very High'],
        'Typical Odds Range': ['1.5-4.0', '1.7-2.2', '1.6-2.4', '5.0-100+'],
        'Liquidity': ['Very High', 'High', 'High', 'Medium'],
    }
)

print(betting_markets)


In [ ]:
def decimal_payout(stake, odds):
    total_return = stake * odds
    return total_return, total_return - stake


def odds_to_probability(decimal_odds):
    return 1 / decimal_odds


def calculate_overround(odds_list):
    total_probability = sum(1 / odds for odds in odds_list)
    return total_probability - 1


def calculate_fair_odds(odds_dict):
    total_probability = sum(1 / odds for odds in odds_dict.values())
    fair_probabilities = {
        outcome: (1 / odds) / total_probability
        for outcome, odds in odds_dict.items()
    }
    fair_odds = {
        outcome: 1 / probability
        for outcome, probability in fair_probabilities.items()
    }
    return fair_probabilities, fair_odds


def identify_value_bet(your_probability, bookmaker_odds):
    implied_probability = 1 / bookmaker_odds
    edge = your_probability - implied_probability
    expected_value = your_probability * (bookmaker_odds - 1) - (1 - your_probability)
    return edge > 0, edge, expected_value


In [ ]:
match_odds = {'Home Win': 2.00, 'Draw': 3.50, 'Away Win': 4.00}

for outcome, odds in match_odds.items():
    print(f'{outcome}: odds={odds:.2f}, implied_probability={odds_to_probability(odds):.1%}')

overround = calculate_overround(match_odds.values())
fair_probabilities, fair_odds = calculate_fair_odds(match_odds)
is_value, edge, expected_value = identify_value_bet(0.40, 3.00)

print()
print(f'Overround: {overround:.2%}')
print(f'Fair probabilities sum: {sum(fair_probabilities.values()):.1%}')
print(f'Example value bet: is_value={is_value}, edge={edge:.1%}, ev={expected_value:.3f}')


In [ ]:
odds_range = np.linspace(1.1, 10.0, 200)
implied_probabilities = 1 / odds_range
profits = (odds_range - 1) * 10

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(odds_range, implied_probabilities, linewidth=2)
axes[0].set_title('Odds to Probability')
axes[0].set_xlabel('Decimal Odds')
axes[0].set_ylabel('Implied Probability')
axes[0].grid(True, alpha=0.3)

axes[1].plot(odds_range, profits, linewidth=2, color='coral')
axes[1].set_title('Profit on a $10 Bet')
axes[1].set_xlabel('Decimal Odds')
axes[1].set_ylabel('Profit ($)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Arbitrage Opportunities

Arbitrage appears when the best odds across bookmakers imply a total
probability below 100%.


In [ ]:
def check_arbitrage(odds_home, odds_draw, odds_away):
    total_probability = (1 / odds_home) + (1 / odds_draw) + (1 / odds_away)
    exists = total_probability < 1
    profit_pct = (1 / total_probability - 1) * 100 if exists else 0
    return exists, total_probability, profit_pct


def calculate_arbitrage_stakes(odds_home, odds_draw, odds_away, total_stake=100):
    implied_home = 1 / odds_home
    implied_draw = 1 / odds_draw
    implied_away = 1 / odds_away
    total_probability = implied_home + implied_draw + implied_away
    if total_probability >= 1:
        return None

    stakes = {
        'Home Win': total_stake * implied_home / total_probability,
        'Draw': total_stake * implied_draw / total_probability,
        'Away Win': total_stake * implied_away / total_probability,
    }
    payouts = {
        'Home Win': stakes['Home Win'] * odds_home,
        'Draw': stakes['Draw'] * odds_draw,
        'Away Win': stakes['Away Win'] * odds_away,
    }
    guaranteed_profit = next(iter(payouts.values())) - total_stake
    return stakes, payouts, guaranteed_profit


bookmaker_odds = pd.DataFrame(
    {
        'Outcome': ['Home Win', 'Draw', 'Away Win'],
        'Bookmaker A': [2.10, 3.60, 4.20],
        'Bookmaker B': [1.90, 3.80, 4.00],
    }
)

best_odds = {
    'Home Win': float(bookmaker_odds.loc[0, ['Bookmaker A', 'Bookmaker B']].max()),
    'Draw': float(bookmaker_odds.loc[1, ['Bookmaker A', 'Bookmaker B']].max()),
    'Away Win': float(bookmaker_odds.loc[2, ['Bookmaker A', 'Bookmaker B']].max()),
}

arbitrage_exists, total_probability, profit_pct = check_arbitrage(
    best_odds['Home Win'],
    best_odds['Draw'],
    best_odds['Away Win'],
)

print(bookmaker_odds)
print()
print(f'Best odds: {best_odds}')
print(f'Arbitrage exists: {arbitrage_exists}')
print(f'Total implied probability: {total_probability:.1%}')
print(f'Expected profit: {profit_pct:.2f}%')

arbitrage_plan = calculate_arbitrage_stakes(
    best_odds['Home Win'],
    best_odds['Draw'],
    best_odds['Away Win'],
)
print()
print(f'Stake plan: {arbitrage_plan}')


## Simulated Match Data

We create a season-like dataframe with bookmaker odds, a few summary
features, and the actual full-time result. The same dataframe will power
the strategy, Kelly, and machine-learning sections.


In [ ]:
n_matches = 380
home_strength = rng.normal(0.25, 0.75, n_matches)
away_strength = rng.normal(0.00, 0.75, n_matches)
draw_signal = -0.35 - np.abs(home_strength - away_strength)

logits = np.column_stack(
    [
        home_strength - away_strength + 0.25,
        draw_signal,
        away_strength - home_strength,
    ]
)
probabilities = np.exp(logits)
probabilities /= probabilities.sum(axis=1, keepdims=True)

outcomes = np.array(['H', 'D', 'A'])
full_time_results = np.array([
    rng.choice(outcomes, p=probabilities[i]) for i in range(n_matches)
])

p_home, p_draw, p_away = probabilities.T
margin = 1.05

matches_df = pd.DataFrame(
    {
        'MatchNo': np.arange(1, n_matches + 1),
        'B365H': 1 / (p_home * margin),
        'B365D': 1 / (p_draw * margin),
        'B365A': 1 / (p_away * margin),
        'AvgH': 1 / (p_home * 1.03) + rng.normal(0, 0.03, n_matches),
        'MaxH': 1 / (p_home * 1.01) + rng.normal(0.03, 0.04, n_matches),
        'BWCH': 1 / (p_home * 1.04) + rng.normal(0, 0.03, n_matches),
        'IWCH': 1 / (p_home * 1.05) + rng.normal(0, 0.03, n_matches),
        'HST': rng.poisson(2.5 + 4 * p_home),
        'FTR': full_time_results,
    }
)

for column in ['B365H', 'B365D', 'B365A', 'AvgH', 'MaxH', 'BWCH', 'IWCH']:
    matches_df[column] = matches_df[column].clip(lower=1.05)

print(matches_df.head())
print()
print('Result distribution:')
print(matches_df['FTR'].value_counts(normalize=True).round(3))


## Baseline Strategies: Choosing a Side

These baselines help us separate predictive accuracy from profitability.


In [ ]:
def run_strategy(matches_df, selection_func, stake=10):
    bets = []
    for _, row in matches_df.iterrows():
        selection = selection_func(row)
        if selection is None:
            continue
        odds = row[f'B365{selection}']
        profit = stake * (odds - 1) if selection == row['FTR'] else -stake
        bets.append(
            {
                'MatchNo': row['MatchNo'],
                'Selection': selection,
                'FTR': row['FTR'],
                'Odds': odds,
                'Stake': stake,
                'Profit': profit,
                'Correct': selection == row['FTR'],
            }
        )
    return pd.DataFrame(bets)


def select_home(_row):
    return 'H'


def select_lowest_odds(row):
    odds_map = {'H': row['B365H'], 'D': row['B365D'], 'A': row['B365A']}
    return min(odds_map, key=odds_map.get)


def select_home_when_favorite(row):
    favorite = select_lowest_odds(row)
    return 'H' if favorite == 'H' else None


def summarize_strategy(results_df):
    total_staked = results_df['Stake'].sum()
    total_profit = results_df['Profit'].sum()
    return {
        'Matches Bet': len(results_df),
        'Accuracy': results_df['Correct'].mean(),
        'Total Profit': total_profit,
        'ROI (%)': 100 * total_profit / total_staked if total_staked else 0,
    }


always_home_results = run_strategy(matches_df, select_home)
lowest_odds_results = run_strategy(matches_df, select_lowest_odds)
home_favorite_results = run_strategy(matches_df, select_home_when_favorite)

comparison = pd.DataFrame(
    {
        'Always Home': summarize_strategy(always_home_results),
        'Lowest Odds': summarize_strategy(lowest_odds_results),
        'Home + Lowest': summarize_strategy(home_favorite_results),
    }
).T

print(comparison)


In [ ]:
def add_cumulative_profit(results_df):
    results_df = results_df.copy()
    results_df['CumulativeProfit'] = results_df['Profit'].cumsum()
    return results_df


always_home_curve = add_cumulative_profit(always_home_results)
lowest_odds_curve = add_cumulative_profit(lowest_odds_results)
home_favorite_curve = add_cumulative_profit(home_favorite_results)

plt.figure(figsize=(12, 6))
plt.plot(always_home_curve['MatchNo'], always_home_curve['CumulativeProfit'], label='Always Home')
plt.plot(lowest_odds_curve['MatchNo'], lowest_odds_curve['CumulativeProfit'], label='Lowest Odds')
plt.plot(home_favorite_curve['MatchNo'], home_favorite_curve['CumulativeProfit'], label='Home + Lowest')
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.title('Cumulative Profit by Strategy')
plt.xlabel('Match Number')
plt.ylabel('Profit ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Kelly Criterion

The Kelly Criterion scales the stake to the estimated edge and the odds.


In [ ]:
def kelly(matches_df, win_probability, on='H'):
    starting_bankroll = 100 * matches_df.shape[0]
    bankroll = starting_bankroll
    roi_history = [0.0]

    for _, row in matches_df.iterrows():
        net_odds = row[f'B365{on}'] - 1
        stake = min(
            max((win_probability - (1 - win_probability) / net_odds) * bankroll, 0),
            bankroll,
        )
        bankroll -= stake
        if row['FTR'] == on:
            bankroll += stake * row[f'B365{on}']
        roi_history.append(round((bankroll - starting_bankroll) / starting_bankroll, 2))

    return roi_history


kelly_history = kelly(matches_df.head(120), win_probability=0.45, on='H')

plt.figure(figsize=(12, 5))
plt.plot(kelly_history, linewidth=2)
plt.title('Kelly Criterion ROI Path')
plt.xlabel('Bet Number')
plt.ylabel('ROI')
plt.grid(True, alpha=0.3)
plt.show()


## An End-to-End Example Using Machine Learning for Betting

This section mirrors the book’s compact machine-learning workflow and
keeps the betting evaluation on held-out matches.


In [ ]:
features = ['AvgH', 'MaxH', 'B365H', 'HST', 'BWCH', 'IWCH']
train_df, test_df = train_test_split(
    matches_df, test_size=0.2, random_state=42, stratify=matches_df['FTR']
)

X_train = train_df[features]
X_test = test_df[features]
y_train = train_df['FTR'].map({'H': 1, 'D': 0, 'A': -1})
y_test = test_df['FTR'].map({'H': 1, 'D': 0, 'A': -1})

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f'Model accuracy: {accuracy:.4f}')


In [ ]:
def calculate_roi(profit, total_stake):
    return 0 if total_stake == 0 else (profit / total_stake) * 100


def settle_bet(row, prediction, stake=100):
    if prediction != row['FTR']:
        return -stake
    odds_column = {'H': 'B365H', 'D': 'B365D', 'A': 'B365A'}[prediction]
    return stake * (row[odds_column] - 1)


def predict_match(row):
    row_features = row[features].astype(float).to_frame().T
    row_scaled = scaler.transform(row_features)
    prediction = model.predict(row_scaled)[0]
    return {1: 'H', 0: 'D', -1: 'A'}[prediction]


def betting_performance(matches_df, prediction_func, stake=100):
    total_bets = 0
    winning_bets = 0
    profit = 0

    for _, row in matches_df.iterrows():
        total_bets += 1
        prediction = prediction_func(row)
        winning_bets += int(prediction == row['FTR'])
        profit += settle_bet(row, prediction, stake)

    win_rate = 100 * winning_bets / total_bets if total_bets else 0
    roi = calculate_roi(profit, total_bets * stake)
    return {
        'Total Bets': total_bets,
        'Winning Bets': winning_bets,
        'Win Rate': win_rate,
        'Profit': profit,
        'ROI': roi,
    }


performance = betting_performance(test_df, predict_match)
print(performance)


In [ ]:
def plot_cumulative_profits(matches_df, prediction_func, stake=100):
    cumulative_profits = []
    current_profit = 0

    for _, row in matches_df.iterrows():
        prediction = prediction_func(row)
        current_profit += settle_bet(row, prediction, stake)
        cumulative_profits.append(current_profit)

    plt.figure(figsize=(12, 6))
    plt.plot(cumulative_profits)
    plt.title('Cumulative Profits Over Time')
    plt.xlabel('Number of Bets')
    plt.ylabel('Cumulative Profit ($)')
    plt.grid(True)
    plt.show()


plot_cumulative_profits(test_df, predict_match)


## Notes

This notebook is the Chapter 9 ground truth for the companion code.
The manuscript keeps only the most important snippets, while this notebook
preserves the full runnable flow.
